# Dunnhumby: M1 조건부 CLV 후보 구별력 진단

기존 seed-42 M1 체크포인트를 그대로 사용합니다. 각 신규상품 정답마다 **같은 상품 인기도 10분위**에서 **M1 점수가 가장 가까운 미구매·비정답 상품 5개**를 대조상품으로 고른 뒤, N 적합도와 V 적합도가 정답을 더 높게 평가하는지 확인합니다.

새 학습·체크포인트 선택·재정렬·final test·holdout은 수행하지 않습니다. 결과는 다음 M2·M3·M4 설계의 기제 진단이며 추천성능 결과가 아닙니다.

추가 진단은 동일한 인기도·M1 점수 조건부 비교쌍에서 `q_C`, `q_N`, `q_V` 상위 20%의 N 후보 구별력을 직접 비교합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '075f212'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip().startswith(REVIEWED_SHA)


In [ ]:
import importlib
import json
import torch
import lightgcn_clv_incremental_candidate_signal_diagnostic as diagnostic
diagnostic = importlib.reload(diagnostic)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert diagnostic.CODE_VERSION == 'clv-incremental-candidate-signal-diagnostic-v3'
cfg = diagnostic.configure_incremental_candidate_signal_diagnostic('dunnhumby')
print(json.dumps(diagnostic.preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
diagnostic = importlib.reload(diagnostic)
cfg = diagnostic.configure_incremental_candidate_signal_diagnostic('dunnhumby')
paths = diagnostic.run_incremental_candidate_signal_diagnostic(cfg)


In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['summary_csv'], dtype={'m1_score_gap_cap_in_user_sd': str})
bootstrap = pd.read_csv(paths['bootstrap_csv'], dtype={'m1_score_gap_cap_in_user_sd': str})
report = json.load(open(paths['json']))
print('1) 전체·CLV 구간·M1 점수차 상한별 조건부 후보 구별력')
display(summary)
print('2) 전체·CLV 구간·M1 점수차 상한별 사용자 bootstrap 95% 구간')
display(bootstrap)
print('3) 대조상품 매칭 품질')
print(json.dumps(report['matching_diagnostics'], ensure_ascii=False, indent=2))
print('4) q_C 자체의 후보 구별력')
print(json.dumps(report['q_c_candidate_diagnostic'], ensure_ascii=False, indent=2))
print('판독: N 신호가 전체 및 고CLV에서 0.25·0.10 SD 상한 모두 95% 구간 하한 0.5를 넘고 H&M에서도 같을 때만 점수매칭·세그먼트 강건성으로 기록합니다.')
print('이 조건을 통과해도 해당 신호를 넣은 M2·M3·M4의 성능개선이 입증되는 것은 아닙니다.')
print('결과 파일:', paths)


print('4) q_C·q_N·q_V 상위 20% N 후보 구별력 비교')
upper_tail = pd.read_csv(paths['upper_tail_csv'], dtype={'m1_score_gap_cap_in_user_sd': str})
display(upper_tail)
